In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
DATA_PATH = "../data/UCI_dataset/"


SIGNALS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]

def load_inertial(split):
    X = np.stack([
        np.loadtxt(f"{DATA_PATH}{split}/Inertial Signals/{s}_{split}.txt")
        for s in SIGNALS
    ], axis=2).astype(np.float32)   # (N, 128, 9)
    y = np.loadtxt(f"{DATA_PATH}{split}/y_{split}.txt", dtype=int) - 1
    return X, y

X_train_raw, y_train_raw = load_inertial('train')
X_test_raw,  y_test_raw  = load_inertial('test')

mean = X_train_raw.mean(axis=(0,1), keepdims=True)
std  = X_train_raw.std(axis=(0,1),  keepdims=True) + 1e-8
X_train_raw = (X_train_raw - mean) / std
X_test_raw  = (X_test_raw  - mean) / std

print(f"Raw train: {X_train_raw.shape}")  # (7352, 128, 9)
print(f"Raw test:  {X_test_raw.shape}")   # (2947, 128, 9)

TypeError: cannot unpack non-iterable NoneType object

In [10]:
import sys
sys.path.insert(0, '../ISWC22-HAR')

from models.TinyHAR import TinyHAR_Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model_tinyhar = TinyHAR_Model(
    input_shape=(1, 1, 9, 128),
    number_class=6,
    filter_num=64,
).to(device)

n_params = sum(p.numel() for p in model_tinyhar.parameters())
print(f"Paramètres TinyHAR : {n_params:,}")

Device: cpu


RuntimeError: Calculated padded input size per channel: (1 x 128). Kernel size: (5 x 1). Kernel size can't be greater than actual input size

In [9]:
def make_loader(X, y, batch_size=256, shuffle=True):
    # X initial: (N, 128, 9)
    # TinyHAR attendu: (N, 1, 9, 128)
    X_4d = X[:, None, :, :]

    ds = TensorDataset(
        torch.FloatTensor(X_4d),
        torch.LongTensor(y)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train_raw, y_train_raw, shuffle=True)
test_loader  = make_loader(X_test_raw,  y_test_raw,  shuffle=False)



optimizer = torch.optim.Adam(model_tinyhar.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5, min_lr=1e-6)
criterion = nn.CrossEntropyLoss()


best_acc, patience_cnt, PATIENCE = 0, 0, 20
history_tinyhar = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, 301):
    model_tinyhar.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model_tinyhar(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct    += (out.argmax(1) == yb).sum().item()
        total      += len(yb)

    model_tinyhar.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model_tinyhar(xb)
            val_loss    += criterion(out, yb).item()
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total   += len(yb)

    t_acc = correct / total
    v_acc = val_correct / val_total
    history_tinyhar['train_loss'].append(train_loss / len(train_loader))
    history_tinyhar['val_loss'].append(val_loss / len(test_loader))
    history_tinyhar['train_acc'].append(t_acc)
    history_tinyhar['val_acc'].append(v_acc)

    scheduler.step(val_loss / len(test_loader))

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train_acc: {t_acc*100:.2f}% | val_acc: {v_acc*100:.2f}%")

    if v_acc > best_acc:
        best_acc = v_acc
        patience_cnt = 0
        torch.save(model_tinyhar.state_dict(), 'best_tinyhar.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stopping à epoch {epoch}")
            break

print(f"\nMeilleure accuracy TinyHAR : {best_acc*100:.2f}%")


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2304x64 and 128x128)